# 01. Explore CSV datasets

Inspect the three source datasets, diagnose the FronkonGames header issue, and prepare Steam App IDs for comparison with SteamIDX.

## 1. Setup

In [ ]:
import csv

import numpy as np
import pandas as pd

from video_game_retention.config import PROJECT_ROOT

## 2. Load the CSV datasets

Load the three source datasets and inspect their dimensions.

In [ ]:
pd.set_option("display.max_columns", None)

tamber_df = pd.read_csv(
    f"{PROJECT_ROOT}/data/raw/tamber_steam-video-games.csv",
    header=None,
    names=["user_id", "game_title", "behavior", "value", "extra"],
)

rabieel_df = pd.read_csv(
    f"{PROJECT_ROOT}/data/raw/rabieelkharoua_online-gaming-behavior-dataset.csv"
)

In [ ]:
fronkon_df = pd.read_csv(
    f"{PROJECT_ROOT}/data/raw/games.csv",
    index_col=False
)

In [ ]:
print("Tamber:", tamber_df.shape)
print("Rabieelkharoua:", rabieel_df.shape)
print("FronkonGames:", fronkon_df.shape)

## 3. Preview the datasets

In [ ]:
tamber_df.head()

In [ ]:
rabieel_df.head()

In [ ]:
fronkon_df.head()

## 4. Investigate the FronkonGames CSV header

In [ ]:
fronkon_df.info()

The initial preview suggests that some values are misaligned with their column names. Inspect the raw CSV header and first record to identify the cause.

In [ ]:
csv_path = f"{PROJECT_ROOT}/data/raw/games.csv"

with open(csv_path, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)

    header = next(reader)
    first_row = next(reader)

print("Header columns:", len(header))
print("First row values:", len(first_row))

In [ ]:
for i, value in enumerate(first_row):
    column = header[i] if i < len(header) else "EXTRA COLUMN"
    print(f"{i}: {column} -> {str(value)[:60]}")

The header appears to combine `Discount` and `DLC count` into one field. Split the header into two columns before reloading the CSV.

In [ ]:
corrected_header = header[:7] + ["Discount", "DLC count"] + header[8:]

fronkon_df = pd.read_csv(
    csv_path, header=0, names=corrected_header, index_col=False
)

In [ ]:
fronkon_df.info()

## 5. Explore the corrected game metadata

Inspect Metacritic scores and locate Grand Theft Auto V Legacy by App ID and name.

In [ ]:
fronkon_df.sort_values(by="Metacritic score", ascending=False).head(5)

In [ ]:
fronkon_df.loc[fronkon_df["AppID"] == 271590].head()

In [ ]:
fronkon_df[fronkon_df["Metacritic score"] == 96]

In [ ]:
fronkon_df.loc[
    fronkon_df["Name"].str.contains("Grand Theft Auto V Legacy", case=False, na=False),
    ["AppID", "Name"],
]

## 6. Extract unique Steam App IDs

Convert IDs to numeric values, discard missing or infinite values, and remove duplicates.

In [ ]:
fronkon_ids = (
    pd.to_numeric(fronkon_df["AppID"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .astype(int)
    .drop_duplicates()
    .tolist()
)

print(fronkon_ids[:10])

In [ ]:
len(fronkon_ids)